# Examen parcial 2 - Diseño de una ruta de seguimiento para un robot SCARA

Cruz Ramírez Dafne Aislinn [Entrega realizadad en equipo con Ramiro Sánchez Leonardo]

Evidencias: https://github.com/leogrief666/Robotica_2026_2/tree/main/Examen%202

## 1. Introducción

   Durante el desarrollo de este examen, se planteó un problema muy real que se vive constantemente en la industria: cuando se diseña o programa un brazo robótico, no solamente basta con lograr que el efecto final se mueva de un punto A a un punto B. Si no conoces bien las limitaciones físicas y matemáticas del robot, puedes terminar pidiéndole movimientos que exijan un esfuerzo inalcanzable para los motores. Lo cual va a provocar que se desgasten prematuramente, den tirones bruscos, o peor aún, que el robot entre en una zona de singularidad donde se vuelve completamente incontrolable y pierde su capacidad de maniobra.

Por esta razón, antes de manufacturar o armar cualquier mecanismo en la vida real, es muy importante evaluar a fondo sus capacidades. Durante este examen nos centramos en el análisis de un manipulador tipo SCARA de tres grados de libertad. La idea principal fue poner a prueba este sistema asignándole una trayectoria simple, pero con un nivel de exigencia alto: llevando la operar muy cerca de la frontera de su espacio de trabajo.

Lo que se busca demostrar con esto es que, utilizando puramente el modelo matemático y la simulación computacional, es completamente posible prever cómo se va a comportar la máquina ante una tarea específica. Calculando el índice de manipulabilidad y extrayendo las gráficas dinámicas de la simulación, podemos conocer de antemano la velocidad, el torque y la energía que van a demandar los actuadores justo en el momento crítico de la operación. A fin de cuentas este análisis nos permite garantizar que los motores seleccionados van a soportar el trabajo de forma eficiente y segura, demostrando que la simulación es un paso indispensable ante cualquier implementación física.


## 2. Desarollo 

   Para abordar los objetivos del examen y estructurar el análisis, se dividió el desarrollo matemático en 4 fases fundamentales:

### 2.1 Propuesta de la trayectoria (lugar geométrico)

   Para poner a prueba la destreza del manipulador coma propusimos una ruta lineal transversal. Considerando que las longitudes de los eslabones son _L1 = 0.5 [m], L2= 0.5 [m] y L3 = 0.25 [m]_ (lo cual suma un alcance máximo absoluto de 1.25 m), establecemos el punto de inicio en una configuración retraída _Pi = (0.5, -0.3)_ y el punto final en _Pf = (1.15, 0.35)_.

Esta selección no la elegimos al azar dos puntos al llegar al punto Pf, el robot se encuentra a _1.20 [m]_ de su base, lo que representa una extensión del 96% de su alcance máximo punto esto obliga al mecanismo a operar peligrosamente cerca de la singularidad de borde, el escenario perfecto para evaluar su pérdida de maniobrabilidad.

### 2.2 Modelo cinemático y de trayectoria

   Para que la herramienta trace una línea recta en el espacio cartesiano, resolvemos la cinemática inversa al instante a instante, mapeando las coordenadas (x, y) a los requerimientos angulares de cada motor (teta 1, teta 2, teta3).

Sin embargo, como exigirle a un robot que siga una línea recta desde el reposo absoluto implica un cambio brusco de velocidad, esto en la vida real requería una aceleración infinita (un tirón mecánico que destruiría los actuadores). Para evitar esto, implementamos una interpolación mediante un polinomio de quinto grado. la ecuación de posición articular está dada por:

<div align="center">
  <img src="Imágenes examen/Form1.png" width="600">
  <br>
  <sub><i>Figura 1: Ecuación de posición articular interpolada mediante un polinomio de quinto grado para garantizar arranques y paradas suaves.</i></sub>
</div>
<br>

La gran ventaja de este polinomio frente a uno de tercer grado es que nos permite garantizar no solo que la velocidad sea cero al inicio y al final del movimiento, sino que la aceleración también parta y termina en cero, asegurando un arranque y un frenado sumamente suaves.

### 2.3 Evaluación cinemática: índice de manipulabilidad

   Para medir matemáticamente qué tan libre es el robot para moverse a lo largo de esta diagonal, se utilizó el índice de manipulabilidad.Este índice se basa en el jacobiano analítico _J(q)_ del sistema: 

<div align="center">
  <img src="Imágenes examen/Form2.png" width="600">
  <br>
  <sub><i>Figura 2: Ecuación de manipulabilidad propuesta por Tsuneo Yoshikawa.</i></sub>
</div>
<br>

Para la arquitectura específica de nuestro SCARA, esta magnitud se simplifica y depende directamente del seno de la segunda articulación. A medida que el brazo se estira para alcanzar el punto final de nuestra diagonal, los eslabones tienden a alinearse. Por lo tanto, esperamos observar una caída drástica en este índice, lo que se comprobará la pérdida de destreza del mecanismo.

### 2.4 Dinámica y potencia de los actuadores

  El análisis cinemático nos da el movimiento, pero para conocer la fuerza real exigida a los motores se aplicó el formalismo dinámico de Euler-Lagrange coma obteniendo el par o torque:
  
<div align="center">
  <img src="Imágenes examen/Form3.png" width="600">
  <br>
  <sub><i>Figura 3: Modelo dinámico basado en el formalismo de Euler-Lagrange para determinar el par mecánico exigido a los actuadores.</i></sub>
</div>
<br>

Donde M(q) representa la matriz de inercia del sistema, C(q, q’) concentra las fuerzas centrípetas y del efecto coriolis, y G(q) representa los componentes gravitacionales. Alimentando este modelo con aceleraciones obtenidas de nuestro polinomio, extraemos el esfuerzo en Newton - metro. 

<div align="center">
  <img src="Imágenes examen/Form4.png" width="600">
  <br>
  <sub><i>Figura 4: Cálculo de la potencia mecánica instantánea en función del par dinámico y la velocidad angular de cada articulación.</i></sub>
</div>
<br>

Finalmente, la potencia mecánica instantánea (P) en watts que debe suministrar cada motor se calcula con el producto de dicho torque y su velocidad angular correspondiente.


  

### 3. Resultados

Una vez que se estableció el puente de comunicación al _ROS 2_ y sincronizamos la matemática de _MATLAB_ con el motor de física de _GAZEBO_ se obtuvieron los siguientes resultados numéricos y visuales para un tiempo de simulación de 10 segundos:

### 3.1 Análisis gráfico

#### 3.1.1 Análisis de la Manipulabilidad vs Tiempo

Tal como se planteó en el desarrollo matemático, la curva de manipulabilidad inicia en un punto estable pero experimenta un decaimiento drástico y continuo conforme el efector final se desplaza por la diagonal. Al acercarnos al segundo 10, el índice se desploma hacia cero validando nuestra hipótesis: extender el robot a 96% de su alcance lo lleva casi a una singularidad geométrica, lo cual reduce severamente su capacidad para reaccionar a perturbaciones o cambiar de dirección.

<div align="center">
  <img src="Imágenes examen/Manipulabilidad.png" width="600">
  <br>
  <sub><i>Figura 5: Índice de manipulabilidad de Yoshikawa. Refleja la pérdida de destreza al aproximarse a la singularidad de borde.</i></sub>
</div>
<br>

#### 3.1.2 Perfiles de velocidad angular y par dinámico

Gracias a la correcta implementación del polinomio de quinto grado, podemos observar que los perfiles de velocidad adoptan formas de campana perfectamente limpias y continuas. Esto se traduce en un movimiento donde no existen picos de aceleración infinitos, permitiendo que el par dinámico exigido a los motores se mantenga dentro de los límites seguros y físicamente realizables, protegiendo la integridad de la transmisión mecánica.

<div align="center">
  <img src="Imágenes examen/Velocidad.png" width="600">
  <br>
  <sub><i>Figura 6: Perfiles de velocidad articular. El polinomio de quinto grado garantiza arranques y paradas suaves.</i></sub>
</div>
<br>

#### 3.1.3 Par dinámico

El esfuerzo mecánico se mantiene dentro de rangos operativos seguros. Los puntos donde las curvas cruzan el cero son críticos, ya que indican en el instante exacto donde los motores dejan de empujar la masa y comienzan a aplicar un frenado dinámico para no rebasar el punto final.

<div align="center">
  <img src="Imágenes examen/Par.png" width="600">
  <br>
  <sub><i>Figura 7: Par dinámico exigido a los actuadores. Los cruces en el eje indican el paso de impulso a frenado.</i></sub>
</div>
<br>

#### 3.1.4 Potencia mecánica total

Esta gráfica nos muestra cuánta energía está gastando realmente el robot durante el recorrido. Se notan muy bien los picos de consumo justo en los momentos de mayor esfuerzo: el primer pico es la energía que se necesita para vencer la inercia y empezar a mover todo el peso del brazo desde cero, mientras que los picos del final representan el esfuerzo eléctrico necesario para frenarlo. Conocer esos valores máximos es súper importante en la vida real, porque con esta información sabemos de qué capacidad comprar los motores para que no se quemen, y qué tipo de fuente de alimentación necesitamos ponerle a nuestra celda de trabajo.

<div align="center">
  <img src="Imágenes examen/Potencia.png" width="600">
  <br>
  <sub><i>Figura 8: Potencia mecánica requerida en Watts. Los picos muestran la energía necesaria para romper y detener la inercia.</i></sub>
</div>
<br>

### 3.2 Evidencia de simulación en Gazebo y RViz2

A continuación se presenta la evidencia en video del comportamiento físico del robot durante la ejecución de los nodos.

<div align="center">
  <a href="https://youtu.be/8XIg9hp6XAE" target="_blank">
    <img src="https://img.youtube.com/vi/8XIg9hp6XAE/hqdefault.jpg" width="600" alt="Video de Simulación SCARA">
  </a>
  <br>
  <sub><i>Video 1: Evidencia de simulación del manipulador SCARA siguiendo la trayectoria diagonal.</i></sub>
</div>

## 4. Conclusión

    Ramiro Sánchez Leonardo
El desarrollo de este examen demostró que la programación y control de los brazos robóticos exigen un análisis integral que va mucho más allá de la simple cinemática de posiciones. A través de la simulación del robot, se pudo comprobar que el uso de herramientas como el polinomio de quinto grado es indispensable en la robótica moderna; esta simple pero poderosa implementación matemática permitió generar un perfil de movimiento suave, protegiendo a los actuadores de aceleraciones infinitas y picos de torque que destruyen la transmisión mecánica en la vida real.


Por otro lado, el índice de manipulabilidad resultó ser un indicador crítico para la seguridad operativa. Se logró comprobar matemáticamente que obligar al robot a estirarse hasta el 96% de su alcance máximo lo acerca peligrosamente a una singularidad de borde, lo cual reduce casi por completo su capacidad de maniobra. Sumando la evaluación dinámica, donde se pudo observar los verdaderos picos de consumo de potencia para romper y detener la inercia del sistema, el examen nos brindó los criterios técnicos exactos que se necesitan en la industria para dimensionar motores y fuentes de alimentación. En conjunto, esto nos permitió confirmar que la simulación es un paso sumamente importante para garantizar el diseño de soluciones mecatrónicas eficientes, seguras y económicamente viables.

    Cruz Ramírez Dafne Aislinn
El desarrollo de este examen demuestra que el control de un manipulador industrial exige un análisis integral que une la planificación cinemática con la viabilidad física del hardware . En primer lugar, la implementación de un polinomio de quinto grado resultó ser una solución matemática indispensable para mitigar el desgaste mecánico ; al forzar que tanto las velocidades como las aceleraciones iniciales y finales terminen en cero, se lograron perfiles continuos en forma de campana que eliminaron los tirones bruscos (jerk) y protegieron las transmisiones .
Por otro lado, la evaluación geométrica mediante el Índice de Manipulabilidad de Yoshikawa evidenció de forma cuantitativa cómo el robot pierde drásticamente su destreza y capacidad de maniobra al aproximarse al segundo 10, validando el riesgo operativo de trabajar en extensiones extremas (96% del alcance máximo) cerca de una singularidad de borde .
Finalmente, el análisis dinámico y de potencia total nos aportó los criterios técnicos reales que se necesitan en la industria para el dimensionamiento de servomotores y fuentes de alimentación, permitiendo mapear con precisión los picos de consumo requeridos para romper la inercia del brazo y ejecutar el frenado dinámico final . En conclusión, esta práctica constata que la simulación es un eslabón mandatorio para garantizar soluciones mecatrónicas eficientes, seguras y económicamente viables antes de cualquier implementación física .

## 5. Fuentes

[1] R. Kelly y V. Santibáñez, Control de Movimiento de Robots Manipuladores, 1.ª ed. Madrid, España: Pearson Educación, 2003.

[2] J. J. Craig, Robótica: Mecánica y Control, 3.ª ed. Naucalpan de Juárez, México: Pearson Educación, 2006.

[3] M. W. Spong, S. Hutchinson, y M. Vidyasagar, Robot Modeling and Control, 1.ª ed. Hoboken, NJ, EE. UU.: John Wiley & Sons, 2005.

[4] B. Siciliano, L. Sciavicco, L. Villani, y G. Oriolo, Robotics: Modelling, Planning and Control. Londres, Reino Unido: Springer-Verlag, 2009.